# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [15]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：data\E Commerce Dataset.xlsx
项目输出目录：c:\Users\杨玉京\OneDrive\桌面\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [16]:
# 在此写下你的答案：
# 1.每条记录代表一位电商客户的多维度业务信息；
# 2.目标变量是`Churn`列；
# 3.CustomerID 是离散身份编号，数值大小无业务意义，不能当作连续数值分析。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [17]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    pass

# TODO：生成清洗前质量报告
# quality_before = build_quality_report(raw_df)
# display(quality_before)
import pandas as pd

def build_quality_report(data):
    """返回字段级数据质量报告。"""
    report = pd.DataFrame()
    # 数据类型
    report["数据类型"] = data.dtypes
    # 缺失数量
    report["缺失数量"] = data.isna().sum()
    # 缺失比例(%)
    report["缺失比例(%)"] = round(data.isna().mean() * 100, 2)
    # 唯一值数量
    report["唯一值数量"] = data.nunique()
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)


,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [18]:
# TODO：完成项目初始审计
# print("完全重复行数：", ...)
# print("CustomerID 重复数量：", ...)
# print(raw_df["Churn"].value_counts())
# print("流失率：", ...)
#
# for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
#     print(f"\n{col}")
#     print(raw_df[col].value_counts())
# TODO: 完成项目初始审计
# 1. 原始数据的完全重复行数
dup_rows = raw_df.duplicated().sum()
print("完全重复行数：", dup_rows)

# 2. CustomerID重复数量（出现次数大于1的ID总数）
cust_dup = (raw_df["CustomerID"].value_counts() > 1).sum()
print("CustomerID 重复数量：", cust_dup)

# 3. Churn的频数和流失率
print(raw_df["Churn"].value_counts())
churn_rate = raw_df["Churn"].mean() * 100
print("流失率：", f"{churn_rate:.2f}%")

# 4. 主要类别字段的频数
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())


完全重复行数： 0
CustomerID 重复数量： 0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 16.84%

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [19]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [20]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    # TODO：创建日志列表 logs
    # TODO：删除完全重复行，并记录日志
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    # TODO：将 Churn 和 Complain 转为整数类型
    # TODO：返回 cleaned_df 与 cleaning_log
    pass
import pandas as pd

# 预先定义常量（根据作业字段自行调整）
NUMERIC_MISSING_COLS = ["Tenure", "WarehouseToHome", "HourSpendOnApp"]
CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {"Mobile Phone": "Mobile", "Phone": "Mobile", "Computer": "PC"},
    "PreferredPaymentMode": {"Debit Card": "Debit", "Credit Card": "Credit", "UPI": "UPI"},
    "PreferedOrderCat": {"Laptop & Accessory": "Electronics", "Mobile": "Electronics", "Fashion": "Fashion"}
}

def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。
    参数：
        data: 原始用户行为 DataFrame
    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # 1. 复制数据，避免覆盖原始数据
    cleaned_df = data.copy()
    logs = []

    # 2. 删除完全重复行
    before = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates()
    after = len(cleaned_df)
    affected = before - after
    logs.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "drop_duplicates()剔除整行重复",
        "处理前记录数": before,
        "处理后记录数": after,
        "影响记录数": affected
    })

    # 3. 数值缺失字段中位数填补
    for col in NUMERIC_MISSING_COLS:
        bef = len(cleaned_df)
        miss_cnt = cleaned_df[col].isna().sum()
        med = cleaned_df[col].median()
        cleaned_df[col] = cleaned_df[col].fillna(med)
        logs.append({
            "处理步骤": f"填充{col}缺失值",
            "处理规则": "数值字段中位数填补",
            "处理前记录数": bef,
            "处理后记录数": bef,
            "影响记录数": miss_cnt
        })

    # 4. 类别标准化映射
    for col, mapping in CATEGORY_MAPPINGS.items():
        bef = len(cleaned_df)
        raw_vals = cleaned_df[col].copy()
        cleaned_df[col] = cleaned_df[col].map(mapping).fillna(cleaned_df[col])
        affect = (raw_vals != cleaned_df[col]).sum()
        logs.append({
            "处理步骤": f"标准化{col}分类",
            "处理规则": f"映射规则{mapping}",
            "处理前记录数": bef,
            "处理后记录数": bef,
            "影响记录数": affect
        })

    # 5. 类型转换：Churn、Complain 转整数
    for col in ["Churn", "Complain"]:
        bef = len(cleaned_df)
        cleaned_df[col] = cleaned_df[col].astype(int)
        logs.append({
            "处理步骤": f"转换{col}数据类型",
            "处理规则": "转为int整数类型",
            "处理前记录数": bef,
            "处理后记录数": bef,
            "影响记录数": 0
        })

    # 生成日志DataFrame
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log


### 任务 3：运行清洗函数并查看日志

In [21]:
# TODO：执行清洗
# cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
#
# display(cleaning_log)
# cleaned_df.head()
# TODO: 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()


,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,drop_duplicates()剔除整行重复,5630,5630,0
1,填充Tenure缺失值,数值字段中位数填补,5630,5630,264
2,填充WarehouseToHome缺失值,数值字段中位数填补,5630,5630,251
3,填充HourSpendOnApp缺失值,数值字段中位数填补,5630,5630,255
4,标准化PreferredLoginDevice分类,"映射规则{'Mobile Phone': 'Mobile', 'Phone': 'Mobil...",5630,5630,5630
5,标准化PreferredPaymentMode分类,"映射规则{'Debit Card': 'Debit', 'Credit Card': 'Cr...",5630,5630,3815
6,标准化PreferedOrderCat分类,"映射规则{'Laptop & Accessory': 'Electronics', 'Mob...",5630,5630,2859
7,转换Churn数据类型,转为int整数类型,5630,5630,0
8,转换Complain数据类型,转为int整数类型,5630,5630,0


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile,3,6.00,Debit,Female,3.00,3,Electronics,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile,1,8.00,UPI,Male,3.00,4,Electronics,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile,1,30.00,Debit,Male,2.00,4,Electronics,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile,3,15.00,Debit,Male,2.00,4,Electronics,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile,1,12.00,CC,Male,3.00,3,Electronics,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [22]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
# TODO：生成 outlier_report（每行对应一个待检查字段）
import pandas as pd

pay_mapping = {
    "CC": "Credit Card",
    "COD": "Cash On Delivery",
    "Credit Card": "Credit Card",
    "Cash On Delivery": "Cash On Delivery",
    "UPI": "UPI",
    "Debit Card": "Debit Card",
    "E wallet": "E wallet"
}
cleaned_df["PreferredPaymentMode"] = cleaned_df["PreferredPaymentMode"].map(pay_mapping)

# 1. 新建 TenureGroup 用户时长分层
tenure_bins = [0, 10, 20, 30, 100]
tenure_labels = ["0-10", "11-20", "21-30", "30+"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels)

# 2. 新建 IsMobileLogin：移动端1，其他0
mobile_list = ["Mobile", "Mobile Phone", "Phone"]
cleaned_df["IsMobileLogin"] = cleaned_df["PreferredLoginDevice"].apply(lambda x: 1 if x in mobile_list else 0)

# 3. 生成异常值报告
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = []
for col in outlier_cols:
    res = iqr_outlier_summary(cleaned_df[col])
    res["字段名"] = col
    outlier_report.append(res)
outlier_report = pd.DataFrame(outlier_report)
display(outlier_report)


,Q1,Q3,下限,上限,候选异常值数量,字段名
0,9.00,20.00,-7.50,36.50,2,WarehouseToHome
1,1.00,3.00,-2.00,6.00,703,OrderCount
2,145.77,196.39,69.84,272.33,438,CashbackAmount


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [23]:
# TODO：完成业务规则检查
# business_rule_report = pd.DataFrame({
#     "规则": [...],
#     "不合规记录数": [...]
# })
# display(business_rule_report)
#
# 处理结论：
import pandas as pd

# 统计各业务违规数量
rule1_cnt = (cleaned_df["Tenure"] < 0).sum()
rule2_cnt = (cleaned_df["WarehouseToHome"] < 0).sum()
rule3_cnt = (cleaned_df["OrderCount"] <= 0).sum()
rule4_cnt = (cleaned_df["CashbackAmount"] < 0).sum()

# 构建业务规则报告
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长小于0",
        "仓库距离小于0",
        "订单数小于或等于0",
        "返现金额小于0"
    ],
    "不合规记录数": [rule1_cnt, rule2_cnt, rule3_cnt, rule4_cnt]
})
display(business_rule_report)

# 处理结论
"""
处理结论：
1. 统计全部4项业务逻辑异常数据条数，无论是否存在违规数据均留存报告用于归档；
2. 数值为负属于业务逻辑错误脏数据，本次分析仅统计记录，暂不做删除；
3. 若存在不合规记录，建模阶段可考虑剔除异常行，或标记为异常特征；无违规则说明数值字段业务逻辑正常。
"""


,规则,不合规记录数
0,使用时长小于0,0
1,仓库距离小于0,0
2,订单数小于或等于0,0
3,返现金额小于0,0


'\n处理结论：\n1. 统计全部4项业务逻辑异常数据条数，无论是否存在违规数据均留存报告用于归档；\n2. 数值为负属于业务逻辑错误脏数据，本次分析仅统计记录，暂不做删除；\n3. 若存在不合规记录，建模阶段可考虑剔除异常行，或标记为异常特征；无违规则说明数值字段业务逻辑正常。\n'

---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [24]:
# TODO：完成最终验收
# quality_after = build_quality_report(cleaned_df)
#
# assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
# assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
# assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
# assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
# assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)
#
# TODO：导出下列文件，使用 utf-8-sig 编码：
# quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
# quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
# cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
# cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
#
# TODO：输出 outlier_report 和 business_rule_report
# TODO：输出交付文件的路径
import os
import pandas as pd
from pathlib import Path

# 1. 定义输出文件夹
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

# 2. 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 3. 校验断言
# 数值缺失字段无空值
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
# 分类标准化校验
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
# 衍生字段存在校验
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)

# 4. 导出交付文件，utf-8-sig编码
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# 5. 输出异常值报告、业务规则报告
print("=====候选异常值报告=====")
display(outlier_report)
print("=====业务规则检查报告=====")
display(business_rule_report)

# 6. 打印所有交付文件路径
print("\n全部交付文件路径：")
for file in os.listdir(OUTPUT_DIR):
    full_path = OUTPUT_DIR / file
    print(full_path)


=====候选异常值报告=====


,Q1,Q3,下限,上限,候选异常值数量,字段名
0,9.00,20.00,-7.50,36.50,2,WarehouseToHome
1,1.00,3.00,-2.00,6.00,703,OrderCount
2,145.77,196.39,69.84,272.33,438,CashbackAmount


=====业务规则检查报告=====


,规则,不合规记录数
0,使用时长小于0,0
1,仓库距离小于0,0
2,订单数小于或等于0,0
3,返现金额小于0,0



全部交付文件路径：
output\cleaning_log.csv
output\data_quality_after.csv
output\data_quality_before.csv
output\day04_project
output\ecommerce_customer_cleaned.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

In [ ]:
#1. 数据质量问题：存在字段缺失值、支付方式等字段类别命名不统一、部分数值字段存在极端异常离群值、还有重复数据。
#2. 处理策略：缺失值按字段业务特性用均值 / 中位数 / 众数填充；类别不一致通过建立映射字典标准化统一格式；异常值用 IQR 法识别后按业务规则截断或剔除。
#3. 清洗后数据补齐了空缺、统一了分类口径、剔除了干扰异常值，满足后续分析的格式与数据合规要求，可直接用于第五天分析。
#4. 异常值截断阈值划定、部分小众支付类别的合并规则、缺失值填充的最终方案，仍需业务人员确认适配实际业务场景。